# 03 — Mini-backtest à exécution différée

> **Avertissement — stratégie-jouet.** Ce notebook explique la chronologie d'un backtest. Il ne constitue ni une stratégie de trading ni une estimation de rendement.

In [ ]:
import polars as pl

from poly_data.analysis.backtest import build_transaction_bars, simulate_next_observation_strategy
from poly_data.io.parquet_store import ParquetStore
from poly_data.notebooks import resolve_notebook_context, source_inventory

ctx = resolve_notebook_context()
store = ParquetStore(ctx.root)
print({'root': str(ctx.root), 'mode': ctx.mode, 'revision': ctx.revision})
source_inventory(store, ['trades', 'markets_current', 'market_outcomes'])

## Construire le signal puis attendre la prochaine observation

Le signal compare le dernier prix journalier à sa moyenne mobile. La fonction de simulation enregistre le signal à *t* et ne remplit qu'à *t+1*. Les frais et le glissement sont appliqués au prix de remplissage.

In [ ]:
trades = store.scan('trades').collect()
market_id = trades.select('market_id').unique().sort('market_id').item(0, 0)
market_trades = trades.filter((pl.col('market_id') == market_id) & (pl.col('nonusdc_side') == 'token1'))
bars = build_transaction_bars(market_trades, seconds=86_400)
bars = (
    bars.with_columns(pl.col('close').rolling_mean(3).alias('moving_average'))
    .drop_nulls('moving_average')
    .with_columns((pl.col('close') > pl.col('moving_average')).cast(pl.Int64).alias('signal'))
)
bars

In [ ]:
split_at = bars.select(pl.col('timestamp').quantile(0.7)).item()
train_bars = bars.filter(pl.col('timestamp') <= split_at)
test_bars = bars.filter(pl.col('timestamp') > split_at)
result = simulate_next_observation_strategy(test_bars, fee_bps=10, slippage_bps=20)
print({'market_id': market_id, 'walk_forward_rows': test_bars.height, 'turnover': result.turnover, 'max_drawdown': result.max_drawdown})
result.fills

In [ ]:
buy_and_hold = test_bars.select((pl.col('close').last() - pl.col('close').first()).alias('one_share_price_change')).item()
summary = result.equity.tail(1).with_columns(pl.lit(buy_and_hold).alias('buy_and_hold_price_change'))
summary

Les courbes utilisent une valeur marquée, pas un règlement. Pour mesurer un PnL réalisé, il faut joindre un résultat officiel et respecter la date de résolution.